# Feature Engineering

This notebook covers:
- IP geolocation merge for Fraud_Data
- Time-based feature creation (hour, day, time_since_signup)
- Edge-case handling for time_since_signup (clip negatives, flag column)
- Transaction velocity features (user/device counts, rolling 24h window)
- One-hot encoding of categorical columns
- Save fraud_processed.csv and creditcard_processed.csv (unscaled)

**Note on scaling:** StandardScaler for creditcard features is intentionally NOT applied here.
It will be fit on X_train only inside `modeling.ipynb` after the train/test split,
to prevent data leakage from test set statistics into training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Note: StandardScaler is NOT imported here for creditcard.
# Scaling is deferred to modeling.ipynb to avoid leakage.

In [ ]:
fraud = pd.read_csv("../data/raw/Fraud_Data.csv")
ip_country = pd.read_csv("../data/raw/IpAddress_to_Country.csv")
credit = pd.read_csv("../data/raw/creditcard.csv")

## 1. Fraud_Data — IP Geolocation Merge

In [ ]:
fraud["signup_time"] = pd.to_datetime(fraud["signup_time"])
fraud["purchase_time"] = pd.to_datetime(fraud["purchase_time"])

In [ ]:
fraud["ip_address"] = fraud["ip_address"].astype(np.int64)
ip_country["lower_bound_ip_address"] = ip_country["lower_bound_ip_address"].astype(np.int64)
ip_country["upper_bound_ip_address"] = ip_country["upper_bound_ip_address"].astype(np.int64)

In [ ]:
fraud = fraud.sort_values("ip_address")
ip_country = ip_country.sort_values("lower_bound_ip_address")

In [ ]:
fraud = pd.merge_asof(
    fraud,
    ip_country,
    left_on="ip_address",
    right_on="lower_bound_ip_address",
    direction="backward"
)

In [ ]:
# Keep only rows where IP falls within the matched range; assign 'Unknown' for unmatched
mask_valid = fraud["ip_address"] <= fraud["upper_bound_ip_address"]
fraud.loc[~mask_valid, "country"] = "Unknown"
fraud["country"] = fraud["country"].fillna("Unknown")

In [ ]:
print(fraud["country"].value_counts().head(20))

In [ ]:
print("Null countries:", fraud["country"].isnull().sum())

## 2. Time Features

In [ ]:
fraud["time_since_signup_raw"] = (
    fraud["purchase_time"] - fraud["signup_time"]
).dt.total_seconds()

In [ ]:
# Edge-case handling: flag rows where time_since_signup was <= 0 (data anomaly)
fraud["time_since_signup_flag"] = (fraud["time_since_signup_raw"] <= 0).astype(int)

# Clip negative values to 0 so the feature is always non-negative
fraud["time_since_signup"] = fraud["time_since_signup_raw"].clip(lower=0)

print("Negative/zero time_since_signup count:", fraud["time_since_signup_flag"].sum())
fraud[["time_since_signup_raw", "time_since_signup", "time_since_signup_flag"]].describe()

In [ ]:
fraud["hour_of_day"] = fraud["purchase_time"].dt.hour

In [ ]:
fraud["day_of_week"] = fraud["purchase_time"].dt.dayofweek

In [ ]:
fraud["is_weekend"] = (fraud["day_of_week"] >= 5).astype(int)

## 3. Velocity Features

In [ ]:
user_tx_count = (
    fraud.groupby("user_id")
    .size()
    .reset_index(name="user_transaction_count")
)
fraud = fraud.merge(user_tx_count, on="user_id", how="left")

In [ ]:
device_tx_count = (
    fraud.groupby("device_id")
    .size()
    .reset_index(name="device_transaction_count")
)
fraud = fraud.merge(device_tx_count, on="device_id", how="left")

In [ ]:
device_user_count = (
    fraud.groupby("device_id")["user_id"]
    .nunique()
    .reset_index(name="users_per_device")
)
fraud = fraud.merge(device_user_count, on="device_id", how="left")

In [ ]:
fraud = fraud.sort_values(["user_id", "purchase_time"])

In [ ]:
fraud["previous_purchase_time"] = (
    fraud.groupby("user_id")["purchase_time"].shift()
)

In [ ]:
fraud["seconds_since_previous_transaction"] = (
    fraud["purchase_time"] - fraud["previous_purchase_time"]
).dt.total_seconds()

In [ ]:
fraud["seconds_since_previous_transaction"] = (
    fraud["seconds_since_previous_transaction"]
    .fillna(fraud["seconds_since_previous_transaction"].median())
)

In [ ]:
# Transaction velocity: number of transactions by same device_id within a rolling 24h window
# Sort by device_id + purchase_time, then use a time-based expanding count
fraud = fraud.sort_values(["device_id", "purchase_time"]).reset_index(drop=True)

def rolling_24h_count(group):
    """Count transactions per device within the 24h window before each transaction."""
    times = group["purchase_time"].values
    counts = []
    for i, t in enumerate(times):
        window_start = t - np.timedelta64(24 * 3600, 's')
        # Count transactions strictly before current one within 24h
        count = np.sum((times[:i] >= window_start) & (times[:i] < t))
        counts.append(count)
    return pd.Series(counts, index=group.index)

fraud["transaction_velocity"] = (
    fraud.groupby("device_id", group_keys=False)
    .apply(rolling_24h_count)
)

print("transaction_velocity stats:")
print(fraud["transaction_velocity"].describe())

## 4. Country Fraud Rate (EDA)

In [ ]:
country_fraud = (
    fraud.groupby("country")["class"]
    .mean()
    .sort_values(ascending=False)
)

country_fraud.head(15).plot(kind="bar", figsize=(12, 5))
plt.title("Highest Fraud Rate Countries")
plt.tight_layout()
plt.show()

Several countries exhibit elevated fraud rates. Country-level information provides useful predictive signal.

## 5. Drop Raw Columns & Encode Categoricals

In [ ]:
columns_to_drop = [
    "signup_time",
    "purchase_time",
    "device_id",
    "ip_address",
    "previous_purchase_time",
    "lower_bound_ip_address",
    "upper_bound_ip_address",
    "time_since_signup_raw",  # raw version no longer needed
]

# Drop only columns that exist (safe guard)
cols_present = [c for c in columns_to_drop if c in fraud.columns]
fraud.drop(columns=cols_present, inplace=True)

In [ ]:
categorical_columns = ["source", "browser", "sex", "country"]

In [ ]:
fraud = pd.get_dummies(
    fraud,
    columns=categorical_columns,
    drop_first=True
)

In [ ]:
print("fraud_processed shape:", fraud.shape)
fraud.head()

## 6. Save fraud_processed.csv

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)

fraud.to_csv("../data/processed/fraud_processed.csv", index=False)
print("Saved fraud_processed.csv")

## 7. CreditCard — Deduplicate Only (No Scaling)

**Important:** StandardScaler is NOT applied here. This prevents data leakage.
The scaler will be fit on `X_train` only inside `modeling.ipynb` after the stratified split,
then used to transform both `X_train` and `X_test`.

In [ ]:
print("Credit before dedup:", credit.shape)
credit = credit.drop_duplicates()
print("Credit after dedup:", credit.shape)

In [ ]:
credit.to_csv("../data/processed/creditcard_processed.csv", index=False)
print("Saved creditcard_processed.csv (unscaled — scaling done in modeling.ipynb)")

In [ ]:
print("fraud_processed shape:", fraud.shape)
print("creditcard_processed shape:", credit.shape)